# Lesson 7.1: Chi-Square Tests for Categories

Test whether ad type (video or banner) is related to buying, using the lesson's 200-person table, and measure the size of the link with Cramér's V.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

## The question

A marketing analyst at an online shop must choose which ad to run next month. She showed a video ad to 100 visitors and a banner ad to 100 others, and noted who bought: 60 of the video group and 45 of the banner group. Is ad type really related to buying, or could a 15-point gap like this appear by chance alone?

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## Step 1: Enter the table of counts

Rows are the two ads and columns are bought or not. A chi-square test works on counts like these, never on percentages. Totals first, so you can see where the expected counts will come from.

In [1]:
import numpy as np  # Load NumPy, the library for arrays of numbers, under the short name np.
from scipy import stats  # Load SciPy's statistics tools, which hold the chi-square test we need.

# rows: video ad, banner ad    columns: bought, did not buy  # A comment that says which row and column is which, so the table stays readable later.
observed = np.array([[60, 40],  # Store the counts as a 2 by 2 array. This first row is the video ad: 60 bought, 40 did not.
                     [45, 55]])  # The second row is the banner ad. Chi-square works on counts like these, never on percentages.

print('row totals   :', observed.sum(axis=1))  # Add across each row (axis=1) to get the group sizes. We expect 100 and 100.
print('column totals:', observed.sum(axis=0))  # Add down each column (axis=0) to see how many bought (105) and did not buy (95) overall.
print('grand total  :', observed.sum())  # The grand total is the number of people. We need it to work out the expected counts.

row totals   : [100 100]
column totals: [105  95]
grand total  : 200


**What this means.** Each ad was shown to 100 people, and 105 of the 200 bought. If ad type did not matter, each group would have about 52 or 53 buyers. The real counts are 60 and 45, so there is a gap to explain.

## Step 2: Run the chi-square test

chi2_contingency works out the expected counts (what we would see if ad type and buying were unrelated), then the chi-square statistic, degrees of freedom and p-value. correction=False gives the plain formula used in the lesson.

In [2]:
res = stats.chi2_contingency(observed, correction=False)  # Run the test of independence. correction=False uses the plain formula from the lesson, and the result holds everything we print.

print('expected counts:')  # Print a heading so the numbers below are labelled.
print(res.expected_freq)  # Expected counts are what each cell would hold if ad type had no link to buying.
print(f'chi-square = {res.statistic:.3f}')  # The chi-square statistic adds up how far the real counts sit from the expected ones.
print(f'df         = {res.dof}')  # Degrees of freedom, (rows - 1) x (columns - 1), tell us which chi-square curve to compare with.
print(f'p-value    = {res.pvalue:.4f}')  # The p-value is how often a gap this big would appear if the ad type made no difference.

expected counts:
[[52.5 47.5]
 [52.5 47.5]]
chi-square = 4.511
df         = 1
p-value    = 0.0337


**What this means.** If ad type had no link to buying, each ad would have 52.5 buyers and 47.5 non-buyers. Chi-square = 4.511 with 1 df gives p = 0.0337. That is below 0.05, so a gap this big would be unusual by luck alone. We have evidence of a link.

## Step 3: Check it by hand

Expected = row total x column total / grand total. Each cell then adds (observed - expected)^2 / expected, and the four pieces sum to the same chi-square.

In [3]:
rows = observed.sum(axis=1)  # Row totals again, because the hand calculation starts from them.
cols = observed.sum(axis=0)  # Column totals, the other ingredient of every expected count.
n = observed.sum()  # The grand total n, the number of people in the table.

expected = np.outer(rows, cols) / n  # Expected count = row total x column total / n. np.outer multiplies every row total by every column total.
pieces = (observed - expected) ** 2 / expected  # For each cell compute (observed - expected) squared, divided by expected. Dividing makes a gap matter more when little was expected.

print(pieces.round(3))  # Show the four pieces, rounded to 3 decimals, to see which cells contribute most.
print(f'sum of pieces = {pieces.sum():.3f}')  # Their sum is the chi-square statistic, which should match scipy's answer.

[[1.071 1.184]
 [1.071 1.184]]
sum of pieces = 4.511


**What this means.** Every cell is 7.5 away from its expected count, but the pieces are 1.071 (buyers) and 1.184 (non-buyers), because a gap counts for more where fewer were expected. The four pieces add to 4.511, the same chi-square as scipy gave.

## Step 4: Measure the effect with Cramér's V

A small p-value says the link is probably real, not how big it is. Cramér's V = sqrt(chi-square / (N x (k - 1))), where k is the smaller of the number of rows and columns. It runs from 0 (no link) to 1 (perfect link).

In [4]:
chi2 = res.statistic  # Take the chi-square value from the test result so we can reuse it.
k = min(observed.shape)  # k is the smaller of the number of rows and columns. Here it is 2.
v = np.sqrt(chi2 / (n * (k - 1)))  # Cramer's V = square root of chi-square / (n x (k - 1)). Dividing by n removes the effect of sample size.

print(f'Cramer V = {v:.3f}')  # Print V to three decimals. 0 means no link and 1 means a perfect link.
print('bought: video', observed[0, 0] / rows[0], ' banner', observed[1, 0] / rows[1])  # Print the buying rate in each group (bought / group size) so the size of the gap is visible in plain terms.

Cramer V = 0.150
bought: video 0.6  banner 0.45


**What this means.** V = 0.150 is a small effect (roughly 0.1 small, 0.3 medium, 0.5 large). In plain words, 60% of the video group bought against 45% of the banner group. The p-value says the link is probably real; V says it is modest, not huge.

## Step 5: Bonus: goodness of fit

A goodness-of-fit test checks one set of counts against a claimed pattern. Here a die was rolled 120 times; chisquare assumes every face is equally likely unless you give it other expected counts.

In [5]:
rolls = np.array([25, 17, 15, 23, 24, 16])  # Store how many times each of the six faces came up in 120 rolls of a die.

fit = stats.chisquare(rolls)  # chisquare is the goodness-of-fit test. With no expected counts given, it assumes all six faces are equally likely.

print(f'expected per face = {rolls.sum() / 6:.0f}')  # The expected count per face is the total rolls divided by 6.
print(f'chi-square = {fit.statistic:.3f}, df = {len(rolls) - 1}')  # Print the statistic and its degrees of freedom, which are the number of categories minus 1.
print(f'p-value    = {fit.pvalue:.4f}')  # Print the p-value: how surprising these counts would be if the die were fair.

expected per face = 20
chi-square = 5.000, df = 5
p-value    = 0.4159


**What this means.** A fair die would show each face 20 times in 120 rolls. Chi-square = 5.000 with 5 df gives p = 0.4159, far above 0.05. Counts like 25 and 15 are ordinary luck, so we have no evidence against a fair die. That is not proof it is fair.

## So, what is the answer?

Yes, there is evidence that ad type is related to buying: 60% of video viewers bought against 45% of banner viewers (chi-square = 4.51, df = 1, p = 0.034), so video looks better. The link is modest (Cramer's V = 0.15). Caution: this shows a link, and only supports 'the video ad causes more buying' if people were shown the ads at random.

## Now you try

A clinic wants fewer missed appointments. It sends a text reminder to 100 patients and phones another 100 patients, then records who turned up. 70 of the text group and 58 of the phone group attended. Is the type of reminder related to attending?

**Data:** observed = [[70, 30], [58, 42]]   # rows: text, phone call   columns: attended, missed

1. Run a chi-square test of independence (no continuity correction) and report the expected counts, chi-square, df and p-value.
2. At the 0.05 level, is reminder type related to attendance?
3. Compute Cramer's V and say how big the link is.

*Hint:* Use the same two functions as the lab: chi2_contingency with correction=False, then V = sqrt(chi-square / (n x (k - 1))).

In [ ]:
# Your turn: write your code here


### Solution (try first, then peek)

In [ ]:
import numpy as np
from scipy import stats

observed = np.array([[70, 30],
                     [58, 42]])
res = stats.chi2_contingency(observed, correction=False)
print('expected:', res.expected_freq.round(1).tolist())
print(f'chi-square = {res.statistic:.3f}, df = {res.dof}, p = {res.pvalue:.4f}')

v = np.sqrt(res.statistic / (observed.sum() * (min(observed.shape) - 1)))
print(f'Cramer V = {v:.3f}')
print('attend rates:', observed[:, 0] / observed.sum(axis=1))

expected: [[64.0, 36.0], [64.0, 36.0]]
chi-square = 3.125, df = 1, p = 0.0771
Cramer V = 0.125
attend rates: [0.7  0.58]


**Answer.** Expected counts are 64 attending and 36 missing in each group. Chi-square = 3.125, df = 1, p = 0.0771, which is above 0.05, so we cannot say reminder type matters: a 12-point gap (70% against 58%) is plausible by chance with 100 patients per group. Cramer's V = 0.125 is a small effect. Not significant does not mean the reminders are equal; a larger study might settle it.

## The same question in R, SPSS and Stata

**R**

```r
ads <- matrix(c(60, 45, 40, 55), nrow = 2,  # Build the 2 by 2 table. c() lists the counts column by column: bought (60, 45) first, then did not buy (40, 55).
              dimnames = list(ad = c("video", "banner"), bought = c("yes", "no")))  # dimnames names the rows and columns so the printed table is easy to read.
test <- chisq.test(ads, correct = FALSE)  # chisq.test runs the test. correct = FALSE turns off Yates' correction so R matches the lesson and the Python.
test  # Typing the name test prints the result: X-squared, df and the p-value.
test$expected  # test$expected prints the expected counts, what we would see if ad and buying were unrelated.
sqrt(test$statistic / (sum(ads) * (min(dim(ads)) - 1)))  # R has no built-in Cramer's V, so we compute it: square root of chi-square / (n x (k - 1)).
chisq.test(c(25, 17, 15, 23, 24, 16))  # Given a plain vector of counts, chisq.test does the goodness-of-fit test, assuming equal chances for each face.
```

How to read it: Look for X-squared = 4.5113, df = 1, p-value = 0.03365 in the first output: below 0.05, so ad type and buying look related. The expected counts are 52.5 and 47.5. The V line shows 0.150 (a small effect). The die test gives p = 0.4159, so no evidence of a loaded die.

**SPSS** (Analyze > Descriptive Statistics > Crosstabs (Statistics: Chi-square, Phi and Cramer's V; Cells: Expected))

```
DATA LIST FREE / ad buy count.
BEGIN DATA
1 1 60  1 2 40  2 1 45  2 2 55
END DATA.
VALUE LABELS ad 1 'video' 2 'banner' / buy 1 'bought' 2 'did not buy'.
WEIGHT BY count.
CROSSTABS
  /TABLES=ad BY buy
  /STATISTICS=CHISQ PHI
  /CELLS=COUNT EXPECTED.
```

How to read it: In Chi-Square Tests, read the Pearson Chi-Square row: value 4.511, df 1, Asymp. Sig. (2-sided) 0.034. Below 0.05, so ad type and buying are related. The Symmetric Measures table gives Cramer's V = 0.150, a small effect. The Crosstabulation shows expected counts of 52.5 and 47.5.

**Stata**

```
clear
input ad buy freq
1 1 60
1 2 40
2 1 45
2 2 55
end
tabulate ad buy [fweight = freq], chi2 V expected
```

How to read it: Find the line Pearson chi2(1) = 4.5113 Pr = 0.034: the p-value is below 0.05, so ad type and buying are related. The line Cramer's V = 0.1502 says the effect is small. Each table cell shows the observed count above its expected count (52.5 or 47.5).